# Regional Design Storm from IDF/RP Maps
## Physical spatial consistency via Areal Reduction Factors

**Theory:** A nationwide RP map gives marginal point-frequency estimates.  
To describe what a T-year storm looks like **spatially**, we must:  
1. Choose a target point or basin centroid  
2. Apply an **Areal Reduction Factor (ARF)** to convert point rainfall → areal rainfall  
3. Impose a **spatial decay kernel** (elliptical isohyetal template)  
4. Distribute depth in time using a **Chicago hyetograph**

**Spatial validity limits:**
| Storm type | Valid radius | Valid area |
|---|---|---|
| Convective (1–3 h) | 15–40 km | 700–5 000 km² |
| Mixed (6–12 h) | 40–80 km | 5 000–20 000 km² |
| Stratiform (24 h) | 60–150 km | 10 000–70 000 km² |

Beyond these distances the ARF relationship (empirically derived from rain-gauge networks)  
loses validity and the spatial template is no longer physically meaningful.

**References:**  
- Svensson & Jones (2010) NERC ARF  
- ISPRA/DPC LSPP methodology (Italy)  
- Keifer & Chu (1957) Chicago hyetograph


## 1. Imports & configuration

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from scipy.optimize import curve_fit
from scipy.stats import genextreme
from tqdm.auto import tqdm

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print('cartopy not found – maps will use pcolormesh')

print('All imports OK')


## 2. User configuration

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
IDF_DIR    = '/home/admin_climatecharted_com/data/MOloch/IDF_results'
OUTPUT_DIR = os.path.join(IDF_DIR, 'regional_design_storm')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── TARGET SITE ───────────────────────────────────────────────────────────────
SITE_NAME = 'Milan'
SITE_LAT  = 45.46
SITE_LON  = 9.19

# ── DESIGN STORM PARAMETERS ───────────────────────────────────────────────────
TARGET_RP       = 100          # years
DURATION_H      = 3            # hours  – must match an idf_<d>h.nc file
CHICAGO_R       = 0.35         # peak position ratio (0=start, 1=end; 0.35 common in Italy)
CHICAGO_DT_MIN  = 5.0          # temporal resolution of hyetograph [minutes]

# ── SPATIAL TEMPLATE ──────────────────────────────────────────────────────────
# Elliptical isohyetal decay:  depth(r) = depth_peak * exp(-r² / (2 σ²))
# σ_km is the e-folding radius — physically linked to storm type and duration.
# Empirical guidance (based on Italian DPC/ISPRA practice and Skaugen 1997):
#   1h  convective  → σ ≈  8–12 km   (valid to ~25 km)
#   3h  mixed       → σ ≈ 15–25 km   (valid to ~50 km)
#   6h  mixed       → σ ≈ 25–40 km   (valid to ~80 km)
#  24h  stratiform  → σ ≈ 50–80 km   (valid to ~150 km)

SIGMA_KM_BY_DURATION = {1: 10, 3: 20, 6: 32, 12: 45, 24: 65}
SIGMA_KM = SIGMA_KM_BY_DURATION.get(DURATION_H, 20)

# Storm ellipse aspect ratio and orientation (1.0 = circular)
ELLIPSE_ASPECT  = 1.3     # major/minor axis ratio (>1 = elongated)
ELLIPSE_ANGLE_DEG = 45.0  # major-axis orientation clockwise from North [°]

# Maximum radius to compute [km] — beyond this ARF < ~0.3, truncate
MAX_RADIUS_KM = 3.0 * SIGMA_KM   # rule of thumb: ~3σ

# ── ARF PARAMETRISATION ───────────────────────────────────────────────────────
# Brath et al. (2003) / Bell (1976) formulation for Italy:
#   ARF(A, D) = 1 - exp(-A / (c1 * D^c2))
# where A = area [km²], D = duration [h].
# Coefficients tuned for Italian Mediterranean climate:
ARF_C1 = 2200.0   # [km²]
ARF_C2 = 0.55     # [-]

# ── IDF power-law parameters (fallback scalars) ───────────────────────────────
# Pixel-wise values from idf_param_a.nc / idf_param_n.nc override these.
# i(t) = a / (t + b)^n  [mm/hr, t in hr]
CHICAGO_A = 0.65
CHICAGO_B = 0.10
CHICAGO_N = 0.72

print(f'Site        : {SITE_NAME} ({SITE_LAT:.3f}N, {SITE_LON:.3f}E)')
print(f'Design storm: RP{TARGET_RP} — {DURATION_H}h duration')
print(f'Spatial σ   : {SIGMA_KM} km  (valid to ~{MAX_RADIUS_KM:.0f} km)')
print(f'Max domain  : {MAX_RADIUS_KM:.0f} km radius ≈ {np.pi*MAX_RADIUS_KM**2/1e4:.0f} × 10⁴ km²')


## 3. Load IDF raster at target RP

In [ ]:
idf_path = os.path.join(IDF_DIR, f'idf_{DURATION_H}h.nc')
if not os.path.exists(idf_path):
    raise FileNotFoundError(
        f'{idf_path} not found. Run MORE_precipitation_IDF_Italy.ipynb first.'
    )

ds_idf = xr.open_dataset(idf_path)
da_idf = ds_idf['return_value'].sel(return_period=TARGET_RP).load()
ds_idf.close()

lat = da_idf['lat'].values
lon = da_idf['lon'].values

# ── extract point depth at the target site ────────────────────────────────────
# Find nearest grid cell
lat_idx = int(np.argmin(np.abs(lat - SITE_LAT)))
lon_idx = int(np.argmin(np.abs(lon - SITE_LON)))

POINT_DEPTH_MM = float(da_idf.isel(lat=lat_idx, lon=lon_idx).values)
print(f'Point depth at {SITE_NAME}: {POINT_DEPTH_MM:.1f} mm  (RP{TARGET_RP}, {DURATION_H}h)')
print(f'Grid cell   : lat={lat[lat_idx]:.3f}, lon={lon[lon_idx]:.3f}')


## 4. Areal Reduction Factor (ARF)

The ARF converts **point rainfall** (what the IDF raster gives) to **areal rainfall**  
(what a catchment of area A experiences on average during the same event).

Formula: `ARF(A, D) = 1 - exp(-A / (c₁ × D^c₂))`

ARF < 1 always; for small A or long D, ARF → 1 (point ≈ areal).


In [ ]:
def arf(area_km2, duration_h, c1=ARF_C1, c2=ARF_C2):
    """Brath-Bell ARF: area [km²], duration [h] → scalar in (0,1]."""
    return 1.0 - np.exp(-area_km2 / (c1 * duration_h**c2))

# ── ARF sensitivity table ─────────────────────────────────────────────────────
print(f'ARF table for {DURATION_H}h storms (Brath-Bell, Italian coefficients):')
print(f'  {"Radius [km]":>12s}  {"Area [km²]":>10s}  {"ARF":>6s}  {"Areal depth [mm]":>18s}')
radii = [5, 10, 20, 30, 50, 75, 100, MAX_RADIUS_KM]
for r in radii:
    A   = np.pi * r**2
    a   = arf(A, DURATION_H)
    dep = a * POINT_DEPTH_MM
    flag = ' ← validity limit' if abs(r - MAX_RADIUS_KM) < 1 else ''
    print(f'  {r:>12.0f}  {A:>10.0f}  {a:>6.3f}  {dep:>18.1f} mm{flag}')

# ── Plot ARF vs radius for all durations ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
r_arr = np.linspace(1, 200, 300)
for d, col in zip([1, 3, 6, 12, 24], ['#e74c3c','#e67e22','#2ecc71','#3498db','#9b59b6']):
    A_arr = np.pi * r_arr**2
    ax.plot(r_arr, arf(A_arr, d), label=f'{d}h', color=col, lw=2)
ax.axvline(MAX_RADIUS_KM, color='k', ls='--', lw=1.2, label=f'Validity limit ({DURATION_H}h) = {MAX_RADIUS_KM:.0f} km')
ax.set_xlabel('Storm radius [km]', fontsize=12)
ax.set_ylabel('ARF  (–)', fontsize=12)
ax.set_title('Areal Reduction Factor vs. storm radius', fontsize=13)
ax.legend(title='Duration', fontsize=9)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'arf_vs_radius.png'), dpi=150)
plt.show()
print('ARF figure saved.')


## 5. Spatial rainfall template (elliptical Gaussian isohyets)

The spatial distribution follows an **elliptical Gaussian decay**:

`depth(x,y) = ARF(A_eff) × point_depth × exp(−r_eff² / (2σ²))`

where `r_eff` is the effective radius in the rotated, scaled ellipse coordinate system.  
The ellipse parameters (aspect ratio, orientation) are approximate; real storms  
show more complex shapes. This template is consistent with the literature for  
**design storm spatial patterns** (Skaugen 1997; Asquith & Famiglietti 2000).


In [ ]:
def make_spatial_template(
    site_lat, site_lon,
    lat_grid, lon_grid,
    sigma_km, aspect, angle_deg,
    point_depth_mm, duration_h,
    arf_c1=ARF_C1, arf_c2=ARF_C2,
    max_radius_km=None,
):
    """
    Return a 2-D array of design-storm areal depths [mm] over the IDF grid.

    Each cell gets the areal depth corresponding to the circular area of
    radius equal to its distance from the site, then modulated by the
    elliptical Gaussian shape.

    Parameters
    ----------
    site_lat, site_lon : float – storm centroid
    lat_grid, lon_grid : 1-D arrays – IDF grid coordinates
    sigma_km           : float – Gaussian e-folding radius [km]
    aspect             : float – major/minor axis ratio
    angle_deg          : float – major-axis azimuth clockwise from North [°]
    point_depth_mm     : float – IDF point depth at centroid [mm]
    duration_h         : int   – storm duration [hours]
    max_radius_km      : float or None – truncation radius [km]
    """
    # approximate km per degree
    KM_PER_LAT = 111.0
    KM_PER_LON = 111.0 * np.cos(np.radians(site_lat))

    LON, LAT = np.meshgrid(lon_grid, lat_grid)
    dx_km = (LON - site_lon) * KM_PER_LON   # positive = east
    dy_km = (LAT - site_lat) * KM_PER_LAT   # positive = north

    # rotate into ellipse principal axes
    theta = np.radians(angle_deg)
    u =  dx_km * np.sin(theta) + dy_km * np.cos(theta)   # along major axis
    v = -dx_km * np.cos(theta) + dy_km * np.sin(theta)   # along minor axis

    # effective isotropic radius in ellipse coordinates
    r_eff = np.sqrt(u**2 + (v * aspect)**2)

    # Gaussian depth field (point depth at centroid)
    depth_field = point_depth_mm * np.exp(-0.5 * (r_eff / sigma_km)**2)

    # Apply ARF: each cell area = (grid spacing in km)^2
    dlat_km = np.abs(np.diff(lat_grid).mean()) * KM_PER_LAT
    dlon_km = np.abs(np.diff(lon_grid).mean()) * KM_PER_LON
    cell_area_km2 = dlat_km * dlon_km
    cell_arf = arf(cell_area_km2, duration_h, arf_c1, arf_c2)
    depth_field *= cell_arf

    if max_radius_km is not None:
        outside = r_eff > max_radius_km
        depth_field[outside] = np.nan

    return depth_field, r_eff


depth_template, r_eff_grid = make_spatial_template(
    SITE_LAT, SITE_LON,
    lat, lon,
    SIGMA_KM, ELLIPSE_ASPECT, ELLIPSE_ANGLE_DEG,
    POINT_DEPTH_MM, DURATION_H,
    max_radius_km=MAX_RADIUS_KM,
)

print(f'Template shape : {depth_template.shape}')
print(f'Centroid depth : {np.nanmax(depth_template):.1f} mm')
print(f'Mean (non-NaN) : {np.nanmean(depth_template):.1f} mm')
print(f'Valid cells    : {np.sum(~np.isnan(depth_template)):,}')


## 6. Map the spatial template

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7),
    subplot_kw={'projection': ccrs.PlateCarree()} if HAS_CARTOPY else {})

LON2, LAT2 = np.meshgrid(lon, lat)

for ax, (data, title) in zip(axes, [
    (depth_template,       f'RP{TARGET_RP} {DURATION_H}h Design Storm — Areal Depth [mm]'),
    (r_eff_grid,           'Effective isohyetal radius from centroid [km]'),
]):
    if HAS_CARTOPY:
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.LAND, facecolor='#f0ede6', zorder=0)
    im = ax.pcolormesh(LON2, LAT2, data, cmap='Blues', shading='auto')
    plt.colorbar(im, ax=ax, shrink=0.7)
    ax.scatter([SITE_LON], [SITE_LAT], c='red', s=80, zorder=5,
               label=SITE_NAME, transform=ax.transData if HAS_CARTOPY else None)

    # draw validity circle
    θ = np.linspace(0, 2*np.pi, 360)
    KM_PER_LON = 111.0 * np.cos(np.radians(SITE_LAT))
    cx = SITE_LON + MAX_RADIUS_KM / KM_PER_LON * np.cos(θ)
    cy = SITE_LAT + MAX_RADIUS_KM / 111.0 * np.sin(θ)
    ax.plot(cx, cy, 'r--', lw=1.5, label=f'Validity limit ({MAX_RADIUS_KM:.0f} km)')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle(f'Regional Design Storm — {SITE_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'design_storm_spatial_{SITE_NAME}_RP{TARGET_RP}_{DURATION_H}h.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Spatial template map saved.')


## 7. Temporal distribution — Chicago hyetograph at each cell

In [ ]:
def chicago_hyetograph(total_mm, duration_h, dt_min, r, a, b, n):
    """
    Chicago (Keifer & Chu 1957) design hyetograph.
    Returns DataFrame with columns: time_min, intensity_mm_hr, depth_mm
    """
    dt_hr  = dt_min / 60.0
    n_pre  = int(round(r * duration_h / dt_hr))
    n_post = int(round((1 - r) * duration_h / dt_hr))

    def P(ta):
        return a * ta / (ta + b)**n   # cumulative depth [mm] for reduced time ta

    ta_pre  = np.arange(1, n_pre + 1) * dt_hr
    dP_pre  = np.diff(P(ta_pre), prepend=0.0)
    pre_int = dP_pre[::-1] / dt_hr   # reverse: max nearest peak

    ta_post  = np.arange(1, n_post + 1) * dt_hr
    dP_post  = np.diff(P(ta_post), prepend=0.0)
    post_int = dP_post / dt_hr

    intensities = np.concatenate([pre_int, post_int])
    comp_depth  = np.sum(intensities) * dt_hr
    if comp_depth > 0:
        intensities *= total_mm / comp_depth   # scale to exact total depth

    times = np.arange(len(intensities)) * dt_min
    return pd.DataFrame({'time_min': times,
                         'intensity_mm_hr': intensities,
                         'depth_mm': intensities * dt_hr})

# ── Generate hyetograph at centroid ──────────────────────────────────────────
hyet = chicago_hyetograph(POINT_DEPTH_MM, DURATION_H,
                           CHICAGO_DT_MIN, CHICAGO_R,
                           CHICAGO_A, CHICAGO_B, CHICAGO_N)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.bar(hyet['time_min'], hyet['intensity_mm_hr'],
        width=CHICAGO_DT_MIN * 0.9, color='steelblue', alpha=0.85)
ax1.set_ylabel('Intensity [mm/hr]', fontsize=11)
ax1.set_title(f'Chicago hyetograph — {SITE_NAME}  |  RP{TARGET_RP}, {DURATION_H}h', fontsize=12)
ax1.grid(alpha=0.3)

ax2.step(hyet['time_min'], hyet['depth_mm'].cumsum(),
         where='post', color='navy', lw=2)
ax2.set_xlabel('Time [min]', fontsize=11)
ax2.set_ylabel('Cumulative depth [mm]', fontsize=11)
ax2.axhline(POINT_DEPTH_MM, color='red', ls='--', label=f'Target: {POINT_DEPTH_MM:.1f} mm')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,
    f'chicago_hyetograph_{SITE_NAME}_RP{TARGET_RP}_{DURATION_H}h.png'), dpi=150)
plt.show()
print(f'Hyetograph total depth: {hyet["depth_mm"].sum():.2f} mm  (target: {POINT_DEPTH_MM:.2f} mm)')


## 8. Spatially distributed hyetograph array

In [ ]:
# For each non-NaN cell in the template, scale the Chicago hyetograph
# by the local areal depth.  This gives a full (time, lat, lon) design storm.

hyet_ref = chicago_hyetograph(1.0, DURATION_H, CHICAGO_DT_MIN,   # unit hyetograph
                               CHICAGO_R, CHICAGO_A, CHICAGO_B, CHICAGO_N)
n_steps  = len(hyet_ref)
n_lat, n_lon = depth_template.shape

# Allocate (time, lat, lon) array
storm_cube = np.full((n_steps, n_lat, n_lon), np.nan, dtype=np.float32)
unit_hyet  = hyet_ref['intensity_mm_hr'].values.astype(np.float32)

valid_mask = ~np.isnan(depth_template)
print(f'Computing spatially distributed hyetograph over {valid_mask.sum():,} cells ...')

# Vectorised: outer product of unit hyetograph and spatial depth field
storm_cube[:, valid_mask] = np.outer(unit_hyet,
                                      depth_template[valid_mask]).astype(np.float32)

print(f'Storm cube shape : {storm_cube.shape}  '
      f'(time_steps={n_steps}, lat={n_lat}, lon={n_lon})')
print(f'Peak intensity   : {np.nanmax(storm_cube):.1f} mm/hr  at t=',
      int(np.nanargmax(np.nanmax(storm_cube, axis=(1,2)))) * CHICAGO_DT_MIN, 'min')

# ── Save as NetCDF ─────────────────────────────────────────────────────────────
times_min = hyet_ref['time_min'].values
ds_out = xr.Dataset({
    'intensity': xr.DataArray(storm_cube,
        dims=['time_min', 'lat', 'lon'],
        coords={'time_min': times_min, 'lat': lat, 'lon': lon},
        attrs={'units': 'mm/hr',
               'description': f'Design storm intensity RP{TARGET_RP} {DURATION_H}h Chicago'}),
    'areal_depth': xr.DataArray(depth_template,
        dims=['lat', 'lon'],
        coords={'lat': lat, 'lon': lon},
        attrs={'units': 'mm',
               'description': f'Areal design depth RP{TARGET_RP} {DURATION_H}h'})
}, attrs={
    'site': SITE_NAME,
    'RP_years': TARGET_RP,
    'duration_h': DURATION_H,
    'sigma_km': SIGMA_KM,
    'validity_radius_km': MAX_RADIUS_KM,
    'ARF_c1': ARF_C1, 'ARF_c2': ARF_C2,
    'Chicago_r': CHICAGO_R,
})

out_path = os.path.join(OUTPUT_DIR,
    f'design_storm_{SITE_NAME}_RP{TARGET_RP}_{DURATION_H}h.nc')
ds_out.to_netcdf(out_path)
print(f'Saved: {out_path}')


## 9. Spatial validity assessment

This cell explicitly documents **where** the design storm template is and is not valid.


In [ ]:
print('=' * 65)
print('SPATIAL VALIDITY ASSESSMENT')
print('=' * 65)
print()
print(f'Storm type    : {DURATION_H}h — ', end='')
if DURATION_H <= 3:
    stype = 'convective/mixed'
elif DURATION_H <= 12:
    stype = 'mixed/mesoscale'
else:
    stype = 'stratiform/synoptic'
print(stype)
print()
print(f'Gaussian σ    : {SIGMA_KM} km')
print(f'Valid radius  : {MAX_RADIUS_KM:.0f} km  (3σ rule)')
print(f'Valid area    : {np.pi * MAX_RADIUS_KM**2 / 1e4:.0f} × 10⁴ km²')
print()
print('Physical limits:')
print('  – ARF empirical fit degrades beyond the validity radius')
print('  – Spatial covariance of IDF fields assumed stationary within domain')
print('  – Storm template is NOT a climatological frequency map;')
print('    it describes what ONE RP event looks like spatially,')
print('    given the storm centroid is at the target site.')
print()
print('NOT physically valid for:')
print('  – Nationwide simultaneous RP exceedance mapping')
print('  – Multi-catchment joint frequency analysis')
print('  – Areas with strong orographic gradients within the domain')
print()
print('RECOMMENDED DOMAIN:')
valid_area = np.pi * MAX_RADIUS_KM**2
print(f'  Circle of radius {MAX_RADIUS_KM:.0f} km around {SITE_NAME}')
print(f'  ≈ {valid_area:.0f} km²  ({valid_area/1e4:.1f} × 10⁴ km²)')
